In [ ]:
import duckdb

con = duckdb.connect("health_enforcement.duckdb")
con.execute("SHOW TABLES").df()


In [ ]:
df_enforcement=con.execute("SELECT * FROM fact_enforcement_actions").df()
df_enforcement

In [ ]:
# 1. Row counts — nothing should explode or disappear
print("Row Counts")
print(f"fact_enforcement_actions: {con.execute('SELECT COUNT(*) FROM fact_enforcement_actions').fetchone()[0]}")
print(f"fact_ltc_narratives:      {con.execute('SELECT COUNT(*) FROM fact_ltc_narratives').fetchone()[0]}")
print(f"enriched_mart:            {con.execute('SELECT COUNT(*) FROM enriched_mart').fetchone()[0]}")

# 2. enriched_mart should match fact_enforcement_actions exactly
print("mart and fact_enforcement")
mart_rows = con.execute("SELECT COUNT(*) FROM enriched_mart").fetchone()[0]
fact_rows = con.execute("SELECT COUNT(*) FROM fact_enforcement_actions").fetchone()[0]
print(f"Mart rows == Fact rows: {mart_rows == fact_rows} ({mart_rows} vs {fact_rows})")

# 4. Nulls in joined columns — count mismatches
print("Mismatches")
print(con.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(CASE WHEN FACILITY_TYPE_DESC IS NULL THEN 1 ELSE 0 END) AS missing_fac_type,
        SUM(CASE WHEN TOP_KEYWORDS IS NULL THEN 1 ELSE 0 END)       AS missing_keywords
    FROM enriched_mart
""").df())

# 5. Spot check a known facility
print("Checkin Fac")
print(con.execute("""
    SELECT FACID, FACILITY_NAME, FACILITY_TYPE_DESC, 
           TOP_KEYWORDS
    FROM enriched_mart
    WHERE TOP_KEYWORDS IS NOT NULL
    LIMIT 5
""").df())

In [ ]:
con.close()

In [ ]:
# print(df_facility_types[df_facility_types.duplicated(subset='VALUE', keep=False)][['VALUE', 'DESCRIPTION']].sort_values('VALUE'))